# 1. LangChain: tools and three interacting agents

**Audience:** beginners learning how agent systems are assembled.

This notebook builds three separate LangChain agents:

- **Manager** — decides what work should be delegated.
- **Analyst** — researches and analyses.
- **Reviewer** — challenges the Analyst's work.

The Analyst and Reviewer can use OpenAI web search, web-page fetch, an in-memory vector database,
simple keyword search, a calculator, file tools and a small shell tool.

The point is not production engineering. The point is to make the moving parts visible.

## Install / update the teaching dependencies

Run these commands from the repository root:

    uv remove langchain langchain-openai
    uv add "langchain>=1.3.2" "langchain-openai>=1.1.11" deepagents beautifulsoup4

You only need `OPENAI_API_KEY` for these notebooks.

## 1. Imports and model setup

The model does the language reasoning. Tools are ordinary functions that the model is allowed to call.

**Memory hook:** *agent = model + instructions + tools*.

In [ ]:
from pathlib import Path
import os
import re
import subprocess

import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv

from langchain.agents import create_agent
from langchain_core.documents import Document
from langchain_core.tools import tool
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

load_dotenv()

MODEL = "gpt-4.1-nano"
EMBEDDING_MODEL = "text-embedding-3-small"

model = ChatOpenAI(
    model=MODEL,
    use_responses_api=True,
)

search_model = ChatOpenAI(
    model=MODEL,
    use_responses_api=True,
)

embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

## 2. A tiny local knowledge base

We deliberately use four small synthetic insurance notes so students can inspect everything.

In [ ]:
documents = [
    Document(
        page_content=(
            "Motor claims inflation rose because repair labour, replacement parts, "
            "vehicle technology and hire-car costs became more expensive. "
            "The insurer responded by increasing pricing and tightening claims controls."
        ),
        metadata={"source": "motor_claims_note.txt"},
    ),
    Document(
        page_content=(
            "A household insurer is testing generative AI to summarise claim notes. "
            "The pilot is intended to reduce administrative work, but human claims handlers "
            "remain responsible for coverage decisions and settlement authority."
        ),
        metadata={"source": "claims_ai_pilot.txt"},
    ),
    Document(
        page_content=(
            "Fraud teams combine rules, anomaly detection and investigator judgement. "
            "An AI assistant may help investigators search past cases, but false positives "
            "can create unnecessary referrals and customer friction."
        ),
        metadata={"source": "fraud_note.txt"},
    ),
    Document(
        page_content=(
            "Personally identifiable information in insurance can include customer names, "
            "email addresses, policy numbers, payment details and claim identifiers. "
            "Sensitive information should be minimised before it is sent to external systems."
        ),
        metadata={"source": "privacy_note.txt"},
    ),
]

vector_store = InMemoryVectorStore(embedding=embeddings)
vector_store.add_documents(documents)

## 3. Compare semantic search and keyword search directly

**Keyword search** looks for matching words.

**Semantic search** uses embeddings: numerical representations of meaning.

Semantic search handles paraphrases well; keyword search can be better for exact terms and identifiers.

In [ ]:
print("SEMANTIC SEARCH")
for doc, score in vector_store.similarity_search_with_score(
    "How could AI help insurance companies investigate suspicious claims?",
    k=4,
):
    print("-", doc.metadata["source"], ": score =", score, ":", doc.page_content)

print("\nKEYWORD SEARCH")
query = "fraud investigator"
words = set(re.findall(r"\w+", query.lower()))

for doc in documents:
    text_words = set(re.findall(r"\w+", doc.page_content.lower()))
    print(doc.metadata["source"], "score =", len(words & text_words))

## 4. Turn Python functions into LangChain tools

`@tool` is the important part. It exposes a normal Python function to the agent.

`web_search` uses **OpenAI's built-in web-search tool**, so students do not need Tavily or Exa accounts.

`fetch_url` is different:

- **search** discovers pages,
- **fetch** downloads one specific page.

> **Safety note:** `run_shell` executes commands on the computer running the notebook. Use only harmless commands in this demo.

In [ ]:
@tool
def web_search(query: str) -> str:
    """Search the live web using OpenAI's built-in web-search tool."""
    response = search_model.invoke(
        query,
        tools=[{"type": "web_search"}],
    )

    lines = []
    for block in response.content_blocks:
        if block.get("type") == "text":
            lines.append(block.get("text", ""))
            for citation in block.get("annotations", []):
                if citation.get("url"):
                    lines.append(f"Source: {citation['url']}")
    return "\n".join(lines)


@tool
def fetch_url(url: str) -> str:
    """Fetch one web page and return readable text from it."""
    html = requests.get(url, timeout=20).text
    text = BeautifulSoup(html, "html.parser").get_text(" ", strip=True)
    return text[:12000]


@tool
def semantic_search(query: str) -> str:
    """Search the small local knowledge base by meaning."""
    matches = vector_store.similarity_search_with_score(query, k=3)
    return "\n\n".join(
        f"score={score:.3f} | {doc.metadata['source']}: {doc.page_content}"
        for doc, score in matches
    )


@tool
def keyword_search(query: str) -> str:
    """Search the local knowledge base using exact query words."""
    words = set(re.findall(r"\w+", query.lower()))
    scored = []

    for doc in documents:
        text_words = set(re.findall(r"\w+", doc.page_content.lower()))
        score = len(words & text_words)
        scored.append((score, doc))

    scored.sort(key=lambda item: item[0], reverse=True)

    return "\n\n".join(
        f"score={score} | {doc.metadata['source']}: {doc.page_content}"
        for score, doc in scored[:3]
    )


@tool
def calculator(expression: str) -> str:
    """Evaluate a simple arithmetic expression such as (85 / 55 - 1) * 100."""
    return str(eval(expression, {"__builtins__": {}}, {}))


@tool
def list_files(folder: str = ".") -> str:
    """List files in a folder."""
    return "\n".join(path.name for path in Path(folder).iterdir())


@tool
def read_file(path: str) -> str:
    """Read a UTF-8 text file."""
    return Path(path).read_text(encoding="utf-8")


@tool
def write_file(path: str, content: str) -> str:
    """Write text to a UTF-8 file."""
    Path(path).write_text(content, encoding="utf-8")
    return f"Wrote {path}"


@tool
def run_shell(command: str) -> str:
    """Run a shell command on this computer. Use only harmless commands in this demo."""
    result = subprocess.run(
        command,
        shell=True,
        capture_output=True,
        text=True,
    )
    return result.stdout or result.stderr

## 5. Test a few tools directly

In [ ]:
print(keyword_search.invoke("claims privacy email"))
print()
print(semantic_search.invoke("What privacy risks might an insurer face when using an AI assistant?"))
print()
print(calculator.invoke("(85 / 55 - 1) * 100"))

## 6. Create the Analyst

The Analyst gets the broadest tool set because its job is to investigate.

In [ ]:
research_tools = [
    web_search,
    fetch_url,
    semantic_search,
    keyword_search,
    calculator,
    list_files,
    read_file,
    write_file,
    run_shell,
]

analyst = create_agent(
    model=model,
    tools=research_tools,
    system_prompt=(
        "You are the Analyst. Research the question carefully. "
        "Use tools when they add evidence. Distinguish local teaching notes "
        "from live web evidence. Keep your answer concise and include source URLs "
        "when web search supplies them."
    ),
)

In [ ]:
result = analyst.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "Using the local knowledge base, explain two benefits and two risks "
            "of generative AI in insurance claims."
        ),
    }]
})

print(result["messages"][-1].text)

## 7. Create the Reviewer

The Reviewer can independently challenge the Analyst instead of merely agreeing.

In [ ]:
reviewer = create_agent(
    model=model,
    tools=[web_search, fetch_url, semantic_search, keyword_search, calculator],
    system_prompt=(
        "You are the Reviewer. Be constructively sceptical. "
        "Check whether claims are supported, whether numbers are consistent, "
        "whether important counterarguments are missing, and whether web evidence is current. "
        "Return PASS if the answer is good enough, otherwise return REVISE followed by specific corrections."
    ),
)

## 8. Make agents callable by another agent

This is the key multi-agent idea. We expose each specialist as a tool.

In [ ]:
@tool
def ask_analyst(question: str) -> str:
    """Ask the Analyst to research and answer a question."""
    result = analyst.invoke({
        "messages": [{"role": "user", "content": question}]
    })
    return result["messages"][-1].text


@tool
def ask_reviewer(draft: str) -> str:
    """Ask the Reviewer to critique a draft answer."""
    result = reviewer.invoke({
        "messages": [{
            "role": "user",
            "content": f"Review this draft:\n\n{draft}",
        }]
    })
    return result["messages"][-1].text

## 9. Create the Manager

The Manager has only the two specialist tools, making delegation obvious.

In [ ]:
manager = create_agent(
    model=model,
    tools=[ask_analyst, ask_reviewer],
    system_prompt=(
        "You are the Manager. For substantive questions, first delegate research "
        "to ask_analyst. Then send the Analyst's draft to ask_reviewer. "
        "If the review says REVISE, ask the Analyst once more with the review comments. "
        "After at most one revision, write the final answer yourself. "
        "Do not skip the Reviewer."
    ),
)

## 10. Run the full Manager → Analyst → Reviewer workflow

In [ ]:
question = (
    "What are three realistic uses of generative AI in insurance claims, "
    "and what controls should an insurer put around them? "
    "Use the local notes and current web evidence where useful."
)

result = manager.invoke({
    "messages": [{"role": "user", "content": question}]
})

print(result["messages"][-1].text)

## 11. Inspect the interaction

In [ ]:
for message in result["messages"]:
    message.pretty_print()

## Takeaways

1. A LangChain tool can be a very small Python function.
2. Search, fetch, keyword retrieval and semantic retrieval solve different problems.
3. One agent can call another agent by wrapping the specialist as a tool.
4. The Manager–Analyst–Reviewer structure works, but we wrote the orchestration ourselves.
5. Notebook 2 shows what changes when a richer agent harness supplies delegation, files and execution.

### Common gotchas

- Do not confuse web **search** with downloading a specific page.
- Semantic search is not automatically better than keyword search.
- Giving every agent every tool can make behaviour less focused.
- A shell tool runs real commands.
- Model outputs and tool choices are probabilistic, so two runs can differ.